# VTT Meeting Transcript → Embedding-Ready Chunks Pipeline (Full Batch Draft)

**Objective:** Convert raw WebVTT meeting transcript files stored in a UC volume into structured chunks suitable for embedding and RAG retrieval. This notebook is intended as a working draft using batch processing with complete overwrites.

## Architecture (Medallion)

| Layer | Table | Purpose |
|-------|-------|--------|
| **Bronze** | `dev.callagent.bronze.vtt_raw` | Raw VTT file content, one row per file |
| **Silver** | `dev.callagent.silver.utterances` | Parsed utterances: speaker, text, PII-redacted, call type labelled |
| **Gold** | `dev.callagent.gold.transcript_chunks` | chunks ready for embedding |

In [0]:
import re
import hashlib
import json
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ── Catalog & Schema ──────────────────────────────────────────────
CATALOG = "dev"
BRONZE_SCHEMA = "callagent_bronze"
SILVER_SCHEMA = "callagent_silver"
GOLD_SCHEMA   = "callagent_gold"

# ── Source Volume ─────────────────────────────────────────────────
VOLUME_ROOT = f"/Volumes/{CATALOG}/callagent/raw/teams_transcripts"

# ── Table Names ──────────────────────────────────────────────────
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.vtt_raw"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.utterances"
GOLD_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.transcript_chunks"

# ── Create schemas if needed ─────────────────────────────────────
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

print("✅ Configuration loaded")
print(f"   Source volume : {VOLUME_ROOT}")
print(f"   Bronze table  : {BRONZE_TABLE}")
print(f"   Silver table  : {SILVER_TABLE}")
print(f"   Gold table    : {GOLD_TABLE}")

## Bronze Layer: Raw VTT Ingestion

Reads `.vtt` files from the volume directory tree (partitioned by `call_date=YYYY-MM-DD`).
Each file becomes one row with the full file content, file path, and metadata.

In [0]:
# ── Batch read all VTT files from the volume ─────────────────────
# Each file is read as a single row with `value` = full file text
raw_files = (
    spark.read.format("text")
    .option("wholetext", "true")
    .load(f"{VOLUME_ROOT}/**/*.vtt")
    .withColumnRenamed("value", "file_content")
    .withColumn("file_path", F.col("_metadata.file_path"))
)

# Extract metadata from partition path and file name
# Generate call_id from filename (deterministic hash of file path)
bronze_df = (
    raw_files
    .withColumn("call_date", F.regexp_extract(F.col("file_path"), r"call_date=(\d{4}-\d{2}-\d{2})", 1).cast("date"))
    # .withColumn("file_name", F.regexp_extract(F.col("file_path"), r"([^/]+)\.vtt$", 1))
    .withColumn("file_name", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.regexp_extract(F.col("file_path"), r"([^/]+)\.vtt$", 1), "%20", " "), "%5B", "["), "%5D", "]"))
    .withColumn("meeting_name", F.regexp_extract(F.col("file_name"), r"(.+)-20\d{6}_\d{6}-Meeting Recording$", 1))
    .withColumn("call_id", F.sha2(F.col("file_path"), 256))
    .withColumn("ingestion_ts", F.current_timestamp())
    .select(
        "call_id",
        "call_date",
        "file_path",
        "file_name",
        "meeting_name",
        "file_content",
        "ingestion_ts",
    )
)

print(f"Raw VTT files found: {bronze_df.count()}")
bronze_df.limit(10).display()

In [0]:
# ── Write bronze table with full refresh ──
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)
print(f"✅ Bronze table written: {BRONZE_TABLE}")

## Silver Layer: Structured Calls

Parses the WebVTT format into one row per call, with utterances (speaker turns) separated by newline characters

PII-Masking: two complementary methods applied sequentially to redact PII
1) regex-based for predictable and deterministic patterns (redacted_text)
2) LLM-based using built-in ai_mask() function for variable patterns (ai_redacted_text)

Call Type Labelling: derived via built-in ai_classify() function aginst a fixed label set

- `call_id`, `call_date`, `file_name` (from bronze)
- `utterances` — `array<struct<utterance_idx, speaker, utterance_text, redacted_text, ai_redacted_text, pii_flagged>>`
- `call_transcript_text` - flattened `"Speaker: text"` transcript (built from `ai_redacted_text`), used as chunking input
- `call_type` - one of 'support', 'sales', 'internal', 'onboarding', 'other', assigned by `ai_classify()` on `call_transcript_text`

In [0]:
# ── VTT Parsing Logic ────────────────────────────────────────────
# WebVTT format:
#   WEBVTT\n\n
#   00:00:00.000 --> 00:00:14.143\n
#   <v Speaker Name>Utterance text</v>\n\n
#   00:00:14.843 --> 00:00:33.700\n
#   <v Speaker Name>More text</v>\n

TS_PATTERN = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})\.(\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})\.(\d{3})"
)
SPEAKER_PATTERN = re.compile(r"<v\s+([^>]+)>(.*?)</v>", re.DOTALL)

# ── PII Redaction Patterns ───────────────────────────────────────
PII_PATTERNS = [
    # Email addresses
    re.compile(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b"),
    # US phone numbers ((XXX) XXX-XXXX or XXX-XXX-XXXX)
    re.compile(r"\(?\b\d{3}\)?[\s.\-]?\d{3}[\s.\-]?\d{4}\b"),
    # Sensitive numbers (groups of 4 digits like the end of payment cards or account ids)
    re.compile(r"\b(?:\d{4}[\s\-]?){3}\d{4}\b"),
    # IPv4 addresses
    re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    # ZIP codes (5 digits, but avoid matching within longer numbers)
    re.compile(r"\b\d{5}(?:\-\d{4})?\b"),
]

def _redact_pii(text: str) -> str:
    """Mask common PII patterns in text with placeholders."""
    if not text:
        return text
    redacted = text
    for pattern in PII_PATTERNS:
        redacted = pattern.sub("[MASKED]", redacted)
    return redacted


def _ts_to_seconds(h: str, m: str, s: str, ms: str) -> float:
    """Convert timestamp components to total seconds."""
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0


def parse_vtt(file_content: str) -> dict:
    """
    Parse a WebVTT transcript string into a single flat, single-line,
    speaker-tagged transcript for one call.

    Returns a dict:
        {
            "transcript_text": "Speaker: text Speaker: text ...",
            "redacted_transcript_text": "<PII-masked version of the above>",
            "pii_flagged": bool,
            "utterance_count": int,
        }
    """
    empty = {
        "transcript_text": "",
        "redacted_transcript_text": "",
        "pii_flagged": False,
        "utterance_count": 0,
    }
    if not file_content or not file_content.strip():
        return empty

    blocks = re.split(r"\n\s*\n", file_content.strip())
    utterance_lines = []

    for block in blocks:
        block = block.strip()
        if not block or block.startswith("WEBVTT"):
            continue

        ts_match = TS_PATTERN.search(block)
        if not ts_match:
            continue

        text_part = block[ts_match.end():].strip()

        speaker_match = SPEAKER_PATTERN.search(text_part)
        if speaker_match:
            speaker = speaker_match.group(1).strip()
            utterance_text = speaker_match.group(2).strip()
        else:
            speaker = "Unknown"
            utterance_text = re.sub(r"<[^>]+>", "", text_part).strip()

        # ── Sanitation, applied BEFORE joining and BEFORE PII masking ──
        # Collapse any internal newlines/repeated whitespace within a
        # single utterance so each speaker turn is itself a clean
        # single line before it becomes one line of the transcript.
        utterance_text = re.sub(r"\s+", " ", utterance_text).strip()

        if utterance_text:
            utterance_lines.append(f"{speaker}: {utterance_text}")

    if not utterance_lines:
        return empty

    # Newline join — one speaker turn per line.
    transcript_text = "\n".join(utterance_lines)

    regex_redacted_text = _redact_pii(transcript_text)
    pii_flagged = regex_redacted_text != transcript_text

    return {
        "transcript_text": transcript_text,
        "regex_redacted_text": regex_redacted_text,
        "pii_flagged": pii_flagged,
        "utterance_count": len(utterance_lines),
    }


# Register as a Spark UDF returning a single struct (one row per call)
CALL_SCHEMA = T.StructType([
    T.StructField("transcript_text",      T.StringType(),  False),
    T.StructField("regex_redacted_text",  T.StringType(),  False),
    T.StructField("pii_flagged",          T.BooleanType(), False),
    T.StructField("utterance_count",      T.IntegerType(), False),
])

parse_vtt_udf = F.udf(parse_vtt, CALL_SCHEMA)

print("✅ VTT parser registered as UDF (one row per call, flat single-line transcript text, regex-based PII redaction)")

In [0]:
# PII masking test on sample file
test_meeting = "[Cascadence Support] Ferro & Vance Logistics — Billing Update"
test_content = spark.table(BRONZE_TABLE).where(f"meeting_name='{test_meeting}'").collect()[0]["file_content"]

test_parsed = parse_vtt(test_content)
print(f"{test_parsed['transcript_text']}\n -> {test_parsed['regex_redacted_text']}\n")

Some PII remains in the text after regex-based masking such as partial credit card numbers, date of birth and full addresses. For patterns like these that are difficult to match via regex, a semantic based masking approach can be used by calling the built-in function [ai_mask()](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_mask).

In [0]:
# LLM-based PII masking using ai_mask()
# ai_mask(content, array('label1','label2',...)) uses a foundation model
# to detect and mask named entities, replacing matched text with [MASKED].

# Address remaining PII from same test case
test_content_df = spark.createDataFrame([test_parsed])

mask_labels = ['birthday', 'credit_card_numbers (last 4 digits)', 'physical_address (not references to addresses)']
labels_sql = ", ".join(f"'{l}'" for l in mask_labels)
test_ai_masked_df = test_content_df.withColumn(
    "ai_redacted_text",
    F.expr(f"ai_mask(regex_redacted_text, array({labels_sql}))")
)
test_ai_masked_df[['regex_redacted_text', 'ai_redacted_text']].display()

In [0]:
# ── Silver: parse VTT into one row per call with PII masked using both regex and ai_mask() ───
# - PII masking using regex patterns followed by ai_mask()
# - call labelling using ai_classify()

# PII Labels for ai_mask()
mask_labels = ['birthday', 'credit_card_numbers (last 4 digits)', 'physical_address (not references to addresses)']
labels_sql = ", ".join(f"'{l}'" for l in mask_labels)

# Call Type Labels (closed set for ai_classify)
call_type_labels = ['support', 'sales', 'internal', 'onboarding', 'other']
call_type_labels_sql = ", ".join(f'"{l}"' for l in call_type_labels)

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("parsed", parse_vtt_udf(F.col("file_content")))
    .select(
        "call_id",
        "call_date",
        "file_name",
        "meeting_name",
        F.col("parsed.transcript_text").alias("transcript_text"),
        F.col("parsed.regex_redacted_text").alias("regex_redacted_text"),
        F.col("parsed.pii_flagged").alias("pii_flagged"),
        F.col("parsed.utterance_count").alias("utterance_count"),
    )
    .withColumn("character_count", F.length(F.col("transcript_text")))
    # use LLM-based approach to mask additional PII, fallback to regex-redacted text if LLM fails
    .withColumn(
        "ai_redacted_text", 
        F.coalesce(
            F.expr(f"ai_mask(regex_redacted_text, array({labels_sql}))"),
            F.col("regex_redacted_text")
        )
    )
    # ai_classify() assigns exactly one label from call_type_labels based on full redacted call transcript
    # output is a VARIANT type containing the label, metadata, and error_message
    .withColumn(
        "call_type_raw",
        F.expr(
            f"""
                ai_classify(
                    concat(
                        ai_redacted_text,
                        ' | Meeting Name: ', meeting_name
                    ),
                    '[{call_type_labels_sql}]',
                    MAP('version', '2.1')
                )
            """
        )
    )
    # retrieve the label from the ai_classify() response
    .withColumn(
        "call_type",
        F.expr("variant_get(call_type_raw, '$.response[0].value', 'string')")
    )
    .drop("call_type_raw")
)
silver_df.limit(10).display()

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)
print(f"✅ Silver table written: {SILVER_TABLE}")
print(f"   Total calls: {spark.table(SILVER_TABLE).count()}")
spark.table(SILVER_TABLE).limit(10).display()

## Gold Layer: Redacted Chunks for Embedding

To prepare the source table for setting up a Databricks AI Search Index and Endpoint, we need to apply a technique called chunking to split the text into small units called chunks. This is to improve RAG quality by allowing a wider variety of examples into the reasoning model and to not exceed the embedding model's context window (this would result in truncated prompts).

In [0]:
%pip install -q langchain-text-splitters
dbutils.library.restartPython()

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500
chunk_overlap = 100

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n", ".", "!", "?", " "],
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    is_separator_regex=False,
)


def chunk_text(text: str, call_id: str) -> list:
    """
    Recursive chunking via LangChain's RecursiveCharacterTextSplitter.
    Separator priority: "\n" (speaker-turn boundary) first, then "."
    (sentence boundary), then "," (clause boundary) as fallback — falls
    back further to a hard character cut internally if none apply.

    Call-boundary safety: this function only ever receives ONE call's
    transcript text at a time (applied per-row on a one-row-per-call
    table), so a chunk can never span two calls by construction.

    Returns: list of chunk dicts matching CHUNK_SCHEMA
    """
    if not text:
        return []

    pieces = text_splitter.split_text(text)

    chunks = []
    for i, piece in enumerate(pieces):
        chunk_id = hashlib.sha256(f"{call_id}:{i}".encode("utf-8")).hexdigest()
        chunks.append({
            "chunk_id":    chunk_id,
            "chunk_index": i,
            "chunk_text":  piece.strip(),
        })
    return chunks


chunk_schema = T.ArrayType(T.StructType([
    T.StructField("chunk_id",     T.StringType(),  False),
    T.StructField("chunk_index",  T.IntegerType(), False),
    T.StructField("chunk_text",   T.StringType(),  False),
]))

chunk_text_udf = F.udf(
    lambda text, call_id: chunk_text(text, call_id),
    chunk_schema,
)

all_chunks_df = (
    spark.table(SILVER_TABLE)
    .withColumn("chunks", chunk_text_udf(F.col("ai_redacted_text"), F.col("call_id")))
    .select(
        "call_id",
        "call_date",
        "call_type",
        F.explode("chunks").alias("chunk"),
    )
    .select(
        F.col("chunk.chunk_id").alias("chunk_id"),
        F.col("chunk.chunk_index").alias("chunk_index"),
        "call_id",
        "call_date",
        "call_type",
        F.col("chunk.chunk_text").alias("chunk_text"),
    )
)

print("✅ Chunking transformation complete — ready to write to gold")
all_chunks_df.limit(10).display()

In [0]:
# Write chunk data to gold table
(
    all_chunks_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"✅ Gold table written: {GOLD_TABLE}")
print(f"   Total chunks: {spark.table(GOLD_TABLE).count()}")
spark.table(GOLD_TABLE).limit(10).display()